# Test: data_preprocessing.py

Este notebook verifica que el script de preprocesamiento funciona correctamente
con el dataset MovieLens 20M antes de hacer merge al repo.

**Requisitos:**
- Tener descargado y descomprimido [MovieLens 20M Dataset](https://grouplens.org/datasets/movielens/20m/)
- El zip se descomprime en una carpeta `ml-20m/` con `ratings.csv` y `movies.csv`

## 0. Configuración

**Ajusta esta ruta** a donde tengas la carpeta `ml-20m` descomprimida:

In [3]:
# ====== AJUSTAR ESTA RUTA ======
ML_PATH = "C:/Users/PabloHG/Documents/MINE/Sistemas de Recomendación/Taller/ml-20m"  # Ruta relativa o absoluta a la carpeta ml-20m
# ================================

import os
from pathlib import Path

ml_path = Path(ML_PATH)
assert (ml_path / "ratings.csv").exists(), f"No se encontró {ml_path / 'ratings.csv'}"
assert (ml_path / "movies.csv").exists(), f"No se encontró {ml_path / 'movies.csv'}"
print(f"Dataset encontrado en: {ml_path.resolve()}")
print(f"Archivos: {[f.name for f in ml_path.iterdir() if f.is_file()]}")

Dataset encontrado en: C:\Users\PabloHG\Documents\MINE\Sistemas de Recomendación\Taller\ml-20m
Archivos: ['movies.csv', 'ratings.csv']


## 1. Exploración del dataset original (Punto 1 del taller)

In [4]:
import pandas as pd
import numpy as np

# Cargar datos originales
print("Cargando ratings.csv (puede tomar ~30 segundos)...")
ratings_full = pd.read_csv(ml_path / "ratings.csv")
movies_full = pd.read_csv(ml_path / "movies.csv")

print(f"\nDataset MovieLens 20M:")
print(f"  Ratings:   {len(ratings_full):>12,}")
print(f"  Usuarios:  {ratings_full['userId'].nunique():>12,}")
print(f"  Películas: {ratings_full['movieId'].nunique():>12,}")
print(f"\nColumnas ratings: {list(ratings_full.columns)}")
print(f"Columnas movies:  {list(movies_full.columns)}")
print(f"\nEjemplo ratings:")
ratings_full.head()

Cargando ratings.csv (puede tomar ~30 segundos)...

Dataset MovieLens 20M:
  Ratings:     20,000,263
  Usuarios:       138,493
  Películas:       26,744

Columnas ratings: ['userId', 'movieId', 'rating', 'timestamp']
Columnas movies:  ['movieId', 'title', 'genres']

Ejemplo ratings:


,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [5]:
print("Ejemplo movies:")
movies_full.head(10)

Ejemplo movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [6]:
# Distribución de ratings
print("Distribución de ratings:")
print(ratings_full["rating"].describe())
print(f"\nConteo por valor:")
print(ratings_full["rating"].value_counts().sort_index())

Distribución de ratings:
count    2.000026e+07
mean     3.525529e+00
std      1.051989e+00
min      5.000000e-01
25%      3.000000e+00
50%      3.500000e+00
75%      4.000000e+00
max      5.000000e+00
Name: rating, dtype: float64

Conteo por valor:
rating
0.5     239125
1.0     680732
1.5     279252
2.0    1430997
2.5     883398
3.0    4291193
3.5    2200156
4.0    5561926
4.5    1534824
5.0    2898660
Name: count, dtype: int64


## 2. Ejecutar el preprocesamiento

In [7]:
# Importar las funciones del script
import sys
sys.path.insert(0, "..")  # Para poder importar data_preprocessing desde notebooks/

from data_preprocessing import (
    load_movielens,
    sample_dataset,
    convert_to_compatible_format,
    split_train_test,
    save_outputs
)

ModuleNotFoundError: No module named 'data_preprocessing'

In [ ]:
# Paso 1: Cargar
ratings_df, movies_df = load_movielens(ML_PATH)

In [ ]:
# Paso 2: Muestrear
sample_df, sample_movies = sample_dataset(ratings_df, movies_df)

In [ ]:
# Paso 3: Convertir formato
sample_df = convert_to_compatible_format(sample_df)

In [ ]:
# Paso 4: Split train/test
train_df, test_df = split_train_test(sample_df)

In [ ]:
# Paso 5: Guardar
save_outputs(train_df, test_df, sample_movies)

## 3. Verificar archivos generados

In [ ]:
# Cargar los archivos generados
DATA_DIR = Path("../movie_recommender/data")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
movies = pd.read_csv(DATA_DIR / "movie.csv")

print("=== train.csv ===")
print(f"Shape: {train.shape}")
print(f"Columns: {list(train.columns)}")
print(train.head())

print(f"\n=== test.csv ===")
print(f"Shape: {test.shape}")
print(f"Columns: {list(test.columns)}")
print(test.head())

print(f"\n=== movie.csv ===")
print(f"Shape: {movies.shape}")
print(f"Columns: {list(movies.columns)}")
print(movies.head())

## 4. Verificar compatibilidad con el código existente

Simulamos exactamente lo que hace `recommender_system.py`:

In [ ]:
# Esto es lo que hace recommender_system.py en la línea 9-13:
matrix = train.pivot_table(index="userId", columns="movieId", values="rating")

print(f"Matriz user-item: {matrix.shape}")
print(f"User IDs: {matrix.index.min()} - {matrix.index.max()}")
print(f"Sparsity: {matrix.isna().sum().sum() / (matrix.shape[0] * matrix.shape[1]):.4f}")
print(f"\nUsuarios en train que también están en test: "
      f"{len(set(train['userId']) & set(test['userId']))} / {test['userId'].nunique()}")

In [ ]:
# Verificar que el lookup de películas funciona (lo que hace recommender_system.py línea 126)
sample_mid = train["movieId"].iloc[0]
title = movies[movies.movieId == sample_mid].title.values[0]
print(f"Movie lookup: movieId={sample_mid} -> '{title}'")

# Verificar varios
for mid in train["movieId"].sample(5, random_state=42):
    t = movies[movies.movieId == mid].title.values[0]
    print(f"  movieId={mid} -> '{t}'")

In [ ]:
# Prueba rápida del modelo existente (cosine similarity)
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(matrix.fillna(0))
sim_df = pd.DataFrame(sim, index=matrix.index, columns=matrix.index)

print(f"Matriz de similitud: {sim_df.shape}")
print(f"Similitud promedio: {sim_df.values[np.triu_indices_from(sim_df.values, k=1)].mean():.4f}")

# Vecinos más similares al usuario 1
user1_sims = sim_df[1].sort_values(ascending=False).iloc[1:6]
print(f"\nTop 5 vecinos del usuario 1:")
for uid, s in user1_sims.items():
    print(f"  Usuario {uid}: similitud = {s:.4f}")

In [ ]:
# Test rápido de predicción para confirmar que todo conecta
user_id = 1
test_user = test[test["userId"] == user_id].head(3)

print(f"Predicciones para usuario {user_id}:")
print(f"{'movieId':>8} {'Actual':>8} {'Título':<40}")
print("-" * 60)
for _, row in test_user.iterrows():
    mid = row["movieId"]
    actual = row["rating"]
    t = movies[movies.movieId == mid].title.values
    title = t[0] if len(t) > 0 else "???"
    print(f"{mid:>8} {actual:>8.1f} {title:<40}")

print(f"\n✅ Todo funciona. Los datos son compatibles con el código existente.")

## 5. Resumen

Si todas las celdas anteriores corrieron sin error, el preprocesamiento está listo.

Archivos generados en `movie_recommender/data/`:
- `train.csv` — columnas: `userId, movieId, rating`
- `test.csv` — columnas: `userId, movieId, rating`  
- `movie.csv` — columnas: `movieId, title, genres`

Estos archivos son exactamente lo que espera `recommender_system.py`, `model_evaluator.py` y `user_manager.py`.